# Laboratorio 2 - Complejidad y búsqueda de hiperparámetros

**Estudiantes:** Camilo Molina Plata | Samuel Rozen Mogollon
**Curso:** ISIS2611 — Aprendizaje de Máquina  
**Fecha:** Febrero 2026

---

## Contexto

En el Laboratorio 1 se construyó un primer modelo de regresión lineal 
para estimar el CVD Risk Score de pacientes de AlpesHearth, junto con 
un proceso de limpieza y exploración del dataset. Los resultados 
mostraron correlaciones débiles entre las variables y el target, y los 
scatter plots sugirieron que las relaciones no son lineales.

En este laboratorio se busca fortalecer ese primer acercamiento 
explorando modelos más complejos: regresión polinomial para capturar 
relaciones no lineales, regularización Ridge y Lasso para controlar el 
sobreajuste, y técnicas de selección de hiperparámetros mediante 
validación cruzada. Finalmente se estimará la incertidumbre del mejor 
modelo mediante bootstrapping.

## Dataset

Se trabaja con el dataset resultante del proceso de limpieza del 
Laboratorio 1: 1441 pacientes, 19 variables (18 predictoras + 
CVD Risk Score como target). El dataset fue exportado como 
dataset_limpio_lab2.csv.

## 0. Importación de librerías y dataset

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Base para transformadores personalizados
from sklearn.base import BaseEstimator, TransformerMixin

# Preprocesamiento
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PolynomialFeatures

# Modelos
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Pipeline
from sklearn.pipeline import Pipeline

# Selección de modelos
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, validation_curve

# Métricas
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Imputación
from sklearn.impute import SimpleImputer

# Encoding
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

## 1. Carga del dataset limpio

In [ ]:
df = pd.read_csv('../Datos/dataset_limpio_lab2.csv')

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nNulos por columna:")
print(df.isnull().sum())

## 2. Split train/test

### Decisión de split

Se divide el dataset en **80% entrenamiento (1152 filas)** y 
**20% test (289 filas)** con random_state=42 para garantizar 
reproducibilidad.

Se optó por un split 80/20 en lugar de 70/20/10 porque la búsqueda 
de hiperparámetros se realiza mediante validación cruzada dentro de 
GridSearchCV. Esto significa que GridSearchCV divide internamente el 
conjunto de entrenamiento en k folds, usando k-1 para entrenar y 1 
para validar en cada iteración, rotando hasta cubrir todos los datos. 
Por tanto, el conjunto de validación está implícito en este proceso y 
no es necesario separarlo manualmente.

El 20% de test se reserva exclusivamente para la evaluación final 
de cada modelo y no se usa en ningún momento durante la búsqueda 
de hiperparámetros ni el entrenamiento, garantizando así una 
estimación imparcial del desempeño de generalización.

In [ ]:
# Split train/test
X = df.drop(columns=['CVD Risk Score']) # Variables predictoras
y = df['CVD Risk Score'] # Variable objetivo

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, # 20% para test, 80% para train
    random_state=42 # Para reproducibilidad
)

print(f"Total filas:   {len(df)}")
print(f"Train:         {X_train.shape}")
print(f"Test:          {X_test.shape}")
print(f"\ny_train media: {y_train.mean():.4f}")
print(f"y_test media:  {y_test.mean():.4f}")

> ### Observaciones del split

El split resultó en 1152 filas para entrenamiento y 289 para test, 
consistente con la proporción 80/20 definida. Las medias del target 
en train (18.0055) y test (16.9323) son ligeramente diferentes, lo 
cual es normal en splits aleatorios con datasets de este tamaño y 
no representa un problema para el modelado.

## 3. Baseline Model

Antes de construir cualquier modelo, se establece un punto de 
referencia mínimo llamado baseline. Este consiste en predecir 
siempre la media del CVD Risk Score del conjunto de entrenamiento 
para todos los pacientes del test set, independientemente de sus 
variables clínicas.

El baseline no es un modelo útil en sí mismo, sino una vara de 
medida. Cualquier modelo de machine learning debe superar este 
umbral mínimo para considerarse que está aprendiendo información 
útil de los datos. Si un modelo sofisticado obtiene un error mayor 
al baseline, significa que no está generalizando correctamente y 
algo está mal en el pipeline.

Por definición matemática, el R² del baseline es siempre 0, ya que 
R² mide exactamente qué tanto mejora un modelo sobre predecir la 
media. Un R² positivo indica que el modelo supera el baseline, 
mientras que un R² negativo indica que es peor que predecir siempre 
la media.

In [ ]:
# Baseline model
y_pred_baseline = np.full(len(y_test), y_train.mean())

rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
mae_baseline  = mean_absolute_error(y_test, y_pred_baseline)
r2_baseline   = r2_score(y_test, y_pred_baseline)

print("=== Baseline Model ===")
print(f"RMSE: {rmse_baseline:.4f}")
print(f"MAE:  {mae_baseline:.4f}")
print(f"R²:   {r2_baseline:.4f}")

> ### Resultados del baseline

| Métrica | Valor |
|:--------|------:|
| RMSE    | 3.9784 |
| MAE     | 2.7386 |
| R²      | -0.0785 |

El R² negativo se explica porque la media del target en train (18.0055) 
es sistemáticamente más alta que la media del target en test (16.9323), 
por lo que predecir siempre 18.0 introduce un sesgo hacia arriba en 
todas las predicciones del test set.

El RMSE de 3.97 establece el umbral mínimo de comparación: cualquier 
modelo construido en este laboratorio debe obtener un RMSE menor a 
este valor para considerarse que está aprendiendo información útil 
de los datos. En el contexto del CVD Risk Score, donde la mayoría 
de los valores se concentran entre 10 y 30 puntos, un error promedio 
de 4 puntos es significativo.

## 4. Modelo de regresión polinomial

### Plan de preprocesamiento

Antes de construir cualquier modelo se define un pipeline de 
preprocesamiento que garantiza que todas las transformaciones se 
aprenden exclusivamente sobre el conjunto de entrenamiento y se 
aplican de manera consistente al conjunto de test, evitando data 
leakage.

El pipeline se divide en dos ramas según el tipo de variable:

---

### Variables numéricas

El pipeline numérico aplica cuatro pasos en orden:

**Paso 1 — Imputación con FormulaImputer**

Se implementa un imputador personalizado que distingue entre 
variables que pueden calcularse mediante fórmulas médicas y 
variables que se imputan con estadísticas de train:

Variables imputadas con fórmulas médicas:
- BMI → Weight (kg) / Height (m)²
- Waist-to-Height Ratio → Abdominal Circumference / (Height * 100)
- Estimated LDL → Total Cholesterol - HDL - 30 (Friedewald simplificada)

Para estas variables, si el valor falta pero las variables necesarias 
están disponibles, se calcula directamente. Si también faltan las 
variables necesarias, se usa la mediana como fallback.

Variables imputadas con mediana (distribuciones con outliers o sesgo):
- Age, Weight (kg), Total Cholesterol, HDL, Fasting Blood Sugar,
  Systolic BP, Diastolic BP

Variables imputadas con media (distribuciones simétricas sin outliers):
- Height (m), Abdominal Circumference (cm)

**Paso 2 — Clipping con IQRClipper**

Se implementa un clipper personalizado que calcula los límites 
inferior y superior para cada variable usando el rango intercuartílico 
(IQR) del conjunto de entrenamiento:

- Límite inferior = Q1 - 1.5 × IQR
- Límite superior = Q3 + 1.5 × IQR

Los valores fuera de estos límites se recortan al límite más cercano. 
Esto controla el efecto de outliers extremos que son fisiológicamente 
posibles pero estadísticamente alejados del resto. Los límites se 
aprenden solo de train y se aplican igual a test.

**Paso 3 — PolynomialFeatures**

Se generan características polinomiales a partir de las variables 
numéricas ya imputadas y clipeadas. El grado del polinomio es el 
hiperparámetro principal que se buscará mediante GridSearchCV.

**Paso 4 — Escalamiento**

Se escalan todas las features generadas. La estrategia de escalamiento 
(StandardScaler o MinMaxScaler) es el segundo hiperparámetro que se 
buscará mediante GridSearchCV.

---

### Variables categóricas

El pipeline categórico aplica dos pasos:

**Paso 1 — Imputación con moda**

Los valores faltantes se reemplazan con el valor más frecuente de 
cada columna, calculado solo sobre train.

**Paso 2 — OneHotEncoder**

Cada variable categórica se convierte en columnas binarias de 0s y 1s. 
Por ejemplo, Sex se convierte en Sex_Male y Sex_Female. Se usa 
handle_unknown='ignore' para manejar categorías no vistas en train.

---

### Unión y modelo

Ambas ramas se combinan con ColumnTransformer y se agrega 
LinearRegression al final del pipeline.

---

### Búsqueda de hiperparámetros

Se usa GridSearchCV con cv=5 para buscar la mejor combinación de:

| Hiperparámetro | Valores a explorar |
|:---------------|:-------------------|
| Grado polinomial | 1, 2, 3, 4, 5 |
| Escalamiento | StandardScaler, MinMaxScaler |

En total se evaluarán 10 combinaciones, cada una con 5 folds de 
validación cruzada, para un total de 50 entrenamientos. La métrica 
de selección es RMSE promedio en validación cruzada.

### 4.1 Definición de features

In [ ]:
# def crear_features(df):
#     df = df.copy()
#     
#     # Activity Risk = mapeo ordinal
#     activity_map = {'Low': 3, 'Moderate': 2, 'High': 1}
#     df['Activity Risk'] = df['Physical Activity Level'].map(activity_map)
#     
#     # Smoker_Diabetic = fumador × diabético
#     df['Smoker_Diabetic'] = (
#         (df['Smoking Status'] == 'Yes') & 
#         (df['Diabetes Status'] == 'Yes')
#     ).astype(int)
#     
#     # Categóricas derivadas
#     df['Age Group'] = pd.cut(df['Age'],
#                              bins=[0, 35, 55, 70, 200],
#                              labels=['Joven', 'Adulto', 'Adulto Mayor', 'Senior'])
#     
#     df['BMI Category'] = pd.cut(df['BMI'],
#                                 bins=[0, 18.5, 25, 30, 100],
#                                 labels=['Bajo peso', 'Normal', 'Sobrepeso', 'Obesidad'])
#     
#     df['Glucose Category'] = pd.cut(df['Fasting Blood Sugar (mg/dL)'],
#                                     bins=[0, 100, 126, 1000],
#                                     labels=['Normal', 'Prediabetes', 'Diabetes'])
#     return df

In [ ]:
# X_train = crear_features(X_train)
# X_test  = crear_features(X_test)

### 4.2 Separar columnas numéricas y categóricas

In [ ]:
cols_numericas = [
    'Age', 'Weight (kg)', 'Height (m)', 'BMI',
    'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)',
    'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)',
    'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP',
    'Estimated LDL (mg/dL)'
]

cols_categoricas = [
    'Sex', 'Smoking Status', 'Diabetes Status',
    'Physical Activity Level', 'Family History of CVD',
    'Blood Pressure Category'
]

# cols_numericas = [
#     'Age', 'Weight (kg)', 'Height (m)', 'BMI',
#     'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)',
#     'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)',
#     'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP',
#     'Estimated LDL (mg/dL)',
#     'Activity Risk', 'Smoker_Diabetic'  # agregadas
# ]
# 
# cols_categoricas = [
#     'Sex', 'Smoking Status', 'Diabetes Status',
#     'Physical Activity Level', 'Family History of CVD',
#     'Blood Pressure Category',
#     'Age Group', 'BMI Category', 'Glucose Category'  # agregadas
# ]

### 4.3 Definir clase FormulaImputer para imputación con fórmulas médicas

**Imputación con fórmulas médicas:**
- BMI       -> Weight / Height²  (si falta BMI pero hay Weight y Height)
- Weight    -> BMI * Height²     (si falta Weight pero hay BMI y Height)
- WHR       -> Abdominal / (Height * 100)
- LDL       -> Cholesterol - HDL - 30
- Cholesterol -> LDL + HDL + 30  (si falta Cholesterol pero hay LDL y HDL)
- HDL       -> Cholesterol - LDL - 30 (si falta HDL pero hay Cholesterol y LDL)

**Fallback con mediana:**
Age, Weight, Total Cholesterol, HDL, Fasting Blood Sugar, Systolic BP, Diastolic BP

**Fallback con media:**
Height, Abdominal Circumference

**Fallback final:**
Cualquier NaN restante → mediana

In [ ]:
class FormulaImputer(BaseEstimator, TransformerMixin):

    # Metodo fit calcula las medianas y medias de cada columna excluyendo los valores nulos
    def fit(self, X, y=None): # X es un DataFrame con las columnas numéricas, y es un array de numpy con la variable objetivo (en este caso no se usa pero se incluye por compatibilidad con Pipeline)
        self.medianas_ = np.nanmedian(X, axis=0) # np.nanmedian ignora los valores nulos al calcular la mediana
        self.medias_   = np.nanmean(X, axis=0) # np.nanmean ignora los valores nulos al calcular la media
        return self # El método fit debe retornar self para que el Pipeline pueda encadenar los transformadores correctamente.
    
    def transform(self, X, y=None):
        if hasattr(X, 'values'):  # ← agregar aquí
            X = X.values          # ← agregar aquí
        
        X = X.copy().astype(float)
        
        # Índices de cada columna
        idx_bmi    = cols_numericas.index('BMI')
        idx_weight = cols_numericas.index('Weight (kg)')
        idx_height = cols_numericas.index('Height (m)')
        idx_whr    = cols_numericas.index('Waist-to-Height Ratio')
        idx_abdom  = cols_numericas.index('Abdominal Circumference (cm)')
        idx_ldl    = cols_numericas.index('Estimated LDL (mg/dL)')
        idx_chol   = cols_numericas.index('Total Cholesterol (mg/dL)')
        idx_hdl    = cols_numericas.index('HDL (mg/dL)')
        
        # Columnas con mediana como fallback
        idx_mediana = [
            cols_numericas.index('Age'),
            cols_numericas.index('Weight (kg)'),
            cols_numericas.index('Total Cholesterol (mg/dL)'),
            cols_numericas.index('HDL (mg/dL)'),
            cols_numericas.index('Fasting Blood Sugar (mg/dL)'),
            cols_numericas.index('Systolic BP'),
            cols_numericas.index('Diastolic BP')
        ]
        
        # Columnas con media como fallback
        idx_media = [
            cols_numericas.index('Height (m)'),
            cols_numericas.index('Abdominal Circumference (cm)')
        ]
        
        # =============================================
        # IMPUTACIÓN CON FÓRMULAS MÉDICAS
        # =============================================
        
        # BMI = Weight / Height²
        # Si falta BMI pero hay Weight y Height
        mask = (np.isnan(X[:, idx_bmi]) &
               ~np.isnan(X[:, idx_weight]) &
               ~np.isnan(X[:, idx_height]))
        X[mask, idx_bmi] = X[mask, idx_weight] / X[mask, idx_height] ** 2
        
        # Weight = BMI * Height²
        # Si falta Weight pero hay BMI y Height
        mask = (np.isnan(X[:, idx_weight]) &
               ~np.isnan(X[:, idx_bmi]) &
               ~np.isnan(X[:, idx_height]))
        X[mask, idx_weight] = X[mask, idx_bmi] * X[mask, idx_height] ** 2
        
        # WHR = Abdominal / (Height * 100)
        # Si falta WHR pero hay Abdominal y Height
        mask = (np.isnan(X[:, idx_whr]) &
               ~np.isnan(X[:, idx_abdom]) &
               ~np.isnan(X[:, idx_height]))
        X[mask, idx_whr] = X[mask, idx_abdom] / (X[mask, idx_height] * 100)
        
        # LDL = Cholesterol - HDL - 30
        # Si falta LDL pero hay Cholesterol y HDL
        mask = (np.isnan(X[:, idx_ldl]) &
               ~np.isnan(X[:, idx_chol]) &
               ~np.isnan(X[:, idx_hdl]))
        X[mask, idx_ldl] = X[mask, idx_chol] - X[mask, idx_hdl] - 30
        
        # Cholesterol = LDL + HDL + 30
        # Si falta Cholesterol pero hay LDL y HDL
        mask = (np.isnan(X[:, idx_chol]) &
               ~np.isnan(X[:, idx_ldl]) &
               ~np.isnan(X[:, idx_hdl]))
        X[mask, idx_chol] = X[mask, idx_ldl] + X[mask, idx_hdl] + 30
        
        # HDL = Cholesterol - LDL - 30
        # Si falta HDL pero hay Cholesterol y LDL
        mask = (np.isnan(X[:, idx_hdl]) &
               ~np.isnan(X[:, idx_chol]) &
               ~np.isnan(X[:, idx_ldl]))
        X[mask, idx_hdl] = X[mask, idx_chol] - X[mask, idx_ldl] - 30
        
        # =============================================
        # FALLBACK CON MEDIANA
        # =============================================
        for i in idx_mediana:
            nan_mask = np.isnan(X[:, i])
            X[nan_mask, i] = self.medianas_[i]
        
        # =============================================
        # FALLBACK CON MEDIA
        # =============================================
        for i in idx_media:
            nan_mask = np.isnan(X[:, i])
            X[nan_mask, i] = self.medias_[i]
        
        # =============================================
        # FALLBACK FINAL
        # cubre BMI, WHR, LDL donde también faltaban
        # sus variables fuente
        # =============================================
        for i in range(X.shape[1]):
            nan_mask = np.isnan(X[:, i])
            if nan_mask.any():
                X[nan_mask, i] = self.medianas_[i]
        
        return X

### 4.4 Definir clase IQRClipper para clipping basado en rango intercuartílico

In [ ]:
class IQRClipper(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        Q1 = np.percentile(X, 25, axis=0)
        Q3 = np.percentile(X, 75, axis=0)
        IQR = Q3 - Q1
        self.lower_ = Q1 - 1.5 * IQR
        self.upper_ = Q3 + 1.5 * IQR
        return self
    
    def transform(self, X, y=None):
        return np.clip(X, self.lower_, self.upper_)

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        if hasattr(X, 'values'):
            X = X.values
        X = X.copy().astype(float)
        
        idx_systolic  = cols_numericas.index('Systolic BP')
        idx_diastolic = cols_numericas.index('Diastolic BP')
        idx_chol      = cols_numericas.index('Total Cholesterol (mg/dL)')
        idx_hdl       = cols_numericas.index('HDL (mg/dL)')
        idx_bmi       = cols_numericas.index('BMI')
        idx_activity  = cols_numericas.index('Activity Risk')
        
        # Pulse Pressure = Systolic - Diastolic
        pulse_pressure = X[:, idx_systolic] - X[:, idx_diastolic]
        
        # MAP = Diastolic + Pulse Pressure / 3
        map_ = X[:, idx_diastolic] + pulse_pressure / 3
        
        # Cholesterol Ratio = Total Cholesterol / HDL
        chol_ratio = X[:, idx_chol] / X[:, idx_hdl]
        
        # BMI x Sedentarism = BMI × Activity Risk
        bmi_sedentarism = X[:, idx_bmi] * X[:, idx_activity]
        
        # Agregar nuevas columnas al array
        nuevas = np.column_stack([
            pulse_pressure,
            map_,
            chol_ratio,
            bmi_sedentarism
        ])
        
        return np.hstack([X, nuevas])

### 4.5 Pipeline para variables numericas y categóricas

#### 4.5.1 Pipeline para variables numéricas

In [ ]:
pipeline_numerico = Pipeline([
    ('imputer',  FormulaImputer()),
    ('clipper',  IQRClipper()),
    ('poly',     PolynomialFeatures(include_bias=False)),
    ('scaler',   StandardScaler())
])

# pipeline_numerico = Pipeline([
#     ('imputer',  FormulaImputer()),
#     ('engineer', FeatureEngineer()),  # después de imputar
#     ('clipper',  IQRClipper()),
#     ('poly',     PolynomialFeatures(include_bias=False)),
#     ('scaler',   StandardScaler())
# ])

#### 4.5.2 Pipeline para variables categóricas

In [ ]:
pipeline_categorico = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore'))
])

#### 4.5.3 ColumnTransformer para unir ambas ramas

In [ ]:
preprocessor = ColumnTransformer([
    ('num', pipeline_numerico,   cols_numericas),
    ('cat', pipeline_categorico, cols_categoricas)
])

#### 4.5.4 Pipeline completo con modelo al final

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model',        LinearRegression())
])

### 4.6 GridSearchCV para búsqueda de hiperparámetros

El grado del polinomio p es el hiperparámetro principal del modelo. 
Un grado bajo genera pocas features y puede resultar en underfitting. 
Un grado alto genera demasiadas features y puede resultar en 
overfitting. Para encontrar el grado óptimo se usa GridSearchCV.

GridSearchCV automatiza la búsqueda de hiperparámetros probando 
todas las combinaciones posibles de valores definidos en param_grid 
y evaluando cada una mediante validación cruzada de 5 folds sobre 
el conjunto de entrenamiento. Para cada combinación:

1. Divide X_train en 5 partes iguales
2. Entrena el pipeline con 4 partes
3. Evalúa con la parte restante
4. Repite 5 veces rotando qué parte se usa para validar
5. Promedia los 5 errores obtenidos

Al final selecciona la combinación de hiperparámetros que obtuvo 
el menor RMSE promedio en validación cruzada.

Se exploran las siguientes combinaciones:

| Hiperparámetro | Valores |
|:---------------|:--------|
| Grado polinomial | 1, 2, 3, 4, 5 |
| Escalamiento | StandardScaler, MinMaxScaler |

En total se evalúan 10 combinaciones × 5 folds = 50 entrenamientos.

La métrica de selección es neg_root_mean_squared_error, que es el 
RMSE negativo. GridSearchCV maximiza la métrica por convención, 
entonces usa el negativo del RMSE para que el menor error corresponda 
al mayor valor.

Una vez identificada la mejor combinación, se entrena el modelo 
final con todo X_train y se evalúa una única vez sobre X_test 
reportando RMSE, MAE y R².

In [ ]:
param_grid = {
    'preprocessor__num__poly__degree': [1, 2, 3, 4, 5],
    'preprocessor__num__scaler':       [StandardScaler(), MinMaxScaler(), RobustScaler()]
}

grid_search = GridSearchCV(
    estimator  = pipeline,
    param_grid = param_grid,
    cv         = 5,
    scoring    = 'neg_root_mean_squared_error',
    n_jobs     = -1,
    verbose    = 1
)

grid_search.fit(X_train, y_train)

In [ ]:
print(f"Mejor grado:   {grid_search.best_params_['preprocessor__num__poly__degree']}")
print(f"Mejor scaler:  {grid_search.best_params_['preprocessor__num__scaler']}")
print(f"Mejor RMSE CV: {-grid_search.best_score_:.4f}")

In [ ]:
y_pred_poly = grid_search.best_estimator_.predict(X_test)

rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
mae_poly  = mean_absolute_error(y_test, y_pred_poly)
r2_poly   = r2_score(y_test, y_pred_poly)

print(f"=== Regresión Polinomial - Test ===")
print(f"RMSE: {rmse_poly:.4f}")
print(f"MAE:  {mae_poly:.4f}")
print(f"R²:   {r2_poly:.4f}")

In [ ]:
resultados_cv = pd.DataFrame(grid_search.cv_results_)
print(resultados_cv[['param_preprocessor__num__poly__degree', 
                      'param_preprocessor__num__scaler',
                      'mean_test_score', 
                      'std_test_score']].sort_values('mean_test_score', ascending=False))

> ## Conclusiones Actividad 1 — Regresión Polinomial con GridSearchCV

### Decisiones tomadas

**Variables derivadas:** Inicialmente se intentó enriquecer el dataset 
con features derivadas clínicamente relevantes (Pulse Pressure, MAP, 
Cholesterol Ratio, Activity Risk, Smoker_Diabetic, BMI x Sedentarism) 
dentro del pipeline. Sin embargo, al aumentar el número de variables 
numéricas de 12 a 18, PolynomialFeatures generó un número excesivo 
de features para grados altos, causando overfitting severo desde 
grado 2. Por esta razón se decidió usar únicamente las 12 variables 
numéricas originales para esta actividad. Las features derivadas se 
evaluarán en las actividades de regularización donde Ridge y Lasso 
pueden controlar el overfitting.

**Grados explorados:** 1 a 5. Con 12 variables numéricas, grado 2 
genera ~90 features y grado 3 genera ~560 features. Sin regularización, 
el modelo no puede manejar ese número de features con 1152 filas de 
entrenamiento.

**Scaler:** Se exploraron StandardScaler y MinMaxScaler dentro del 
GridSearchCV.

**Validación cruzada:** cv=5, métrica neg_root_mean_squared_error.

### Resultados

| Grado | Scaler | RMSE CV promedio | Std CV |
|:------|:-------|:----------------:|:------:|
| 1 | StandardScaler | 7.7938 | 0.9952 |
| 1 | MinMaxScaler | 7.7938 | 0.9952 |
| 2 | StandardScaler | 9.6838 | 2.2656 |
| 2 | MinMaxScaler | 9.6838 | 2.2656 |
| 3 | StandardScaler | 31.5184 | 12.881 |
| 3 | MinMaxScaler | 30.8849 | 12.763 |
| 4 | StandardScaler | 151.199 | 26.917 |
| 4 | MinMaxScaler | 146.507 | 32.641 |
| 5 | StandardScaler | 86.911 | 18.858 |
| 5 | MinMaxScaler | 84.521 | 18.235 |

**Mejor configuración:** Grado 1, StandardScaler  
**RMSE CV:** 7.7938  

| Métrica | Test |
|:--------|-----:|
| RMSE | 3.3665 |
| MAE | 1.7778 |
| R² | 0.2278 |

### Observaciones

El grado óptimo resultó ser 1, equivalente a regresión lineal simple. 
Esto es consistente con las correlaciones débiles encontradas en el 
Laboratorio 1, donde ninguna variable superó 0.15 de correlación con 
el target.

El grado 2 obtuvo un RMSE CV de 9.68, cercano al grado 1 pero ya 
mostrando señales de overfitting. A partir de grado 3 el overfitting 
es severo, con RMSE CV de 30 y desviaciones estándar muy altas que 
indican alta varianza entre folds.

La diferencia entre RMSE en CV (7.79) y RMSE en test (3.37) es 
notable. Esto se explica porque el CV divide el train en folds más 
pequeños, reduciendo el tamaño efectivo del conjunto de entrenamiento 
y aumentando el error. El test set al ser evaluado con el modelo 
entrenado sobre todo el train obtiene un error menor.

El modelo supera el baseline (RMSE 3.9784) con un RMSE de 3.3665, 
confirmando que está aprendiendo información útil de los datos. El 
R² de 0.2278 indica que el modelo explica el 22.78% de la variabilidad 
del CVD Risk Score, lo cual es bajo pero consistente con las débiles 
correlaciones del dataset.

Sin regularización, aumentar la complejidad polinomial no mejora el 
modelo. Se espera que la combinación de features polinomiales con 
regularización en la Actividad 4 permita usar grados más altos sin 
overfitting.